In [ ]:
# notebooks/05_interpretation.ipynb

import pandas as pd
import requests
import json

class BiologicalInterpreter:
    """
    Biological interpretation of top genes
    Literature validation and disease relevance
    """
    
    def __init__(self, top_genes, de_results):
        self.top_genes = top_genes
        self.de_results = de_results
        
    def get_gene_info(self, gene_symbol):
        """
        Query NCBI Gene database for gene information
        """
        # This is a simplified example
        # In practice, use proper APIs like mygene.info
        base_url = "https://mygene.info/v3/query"
        
        params = {
            'q': gene_symbol,
            'species': 'human',
            'fields': 'name,summary,pathway,go'
        }
        
        try:
            response = requests.get(base_url, params=params)
            if response.status_code == 200:
                data = response.json()
                if 'hits' in data and len(data['hits']) > 0:
                    return data['hits'][0]
        except:
            pass
        
        return None
    
    def create_top_genes_report(self, top_n=20):
        """
        Create comprehensive report for top genes
        """
        top_gene_list = self.top_genes.head(top_n)['gene'].tolist()
        
        report = []
        
        for gene in top_gene_list:
            # Get differential expression stats
            de_info = self.de_results[self.de_results['gene'] == gene]
            
            gene_report = {
                'Gene': gene,
                'Log2 Fold Change': de_info['log2_fold_change'].values[0] if len(de_info) > 0 else 'N/A',
                'P-value (adjusted)': de_info['p_adjusted'].values[0] if len(de_info) > 0 else 'N/A',
                'Regulation': 'Up' if (len(de_info) > 0 and de_info['log2_fold_change'].values[0] > 0) else 'Down'
            }
            
            # Query gene info (optional, requires internet)
            # gene_info = self.get_gene_info(gene)
            # if gene_info:
            #     gene_report['Description'] = gene_info.get('name', 'N/A')
            
            report.append(gene_report)
        
        report_df = pd.DataFrame(report)
        
        # Save report
        report_df.to_csv('./results/top_genes_report.csv', index=False)
        
        print("=== Top Genes Report ===")
        print(report_df.to_string(index=False))
        
        return report_df
    
    def create_interpretation_summary(self):
        """
        Create summary for IGIB presentation
        """
        summary = f"""
        ╔══════════════════════════════════════════════════════════╗
        ║          BIOLOGICAL INTERPRETATION SUMMARY                ║
        ╚══════════════════════════════════════════════════════════╝
        
        1. TOP DISEASE-ASSOCIATED GENES
        ---------------------------------
        Total significant genes identified: {len(self.de_results[self.de_results['p_adjusted'] < 0.05])}
        
        Top 5 upregulated genes in COPD:
        {self.de_results[self.de_results['log2_fold_change'] > 0].head(5)[['gene', 'log2_fold_change', 'p_adjusted']].to_string(index=False)}
        
        Top 5 downregulated genes in COPD:
        {self.de_results[self.de_results['log2_fold_change'] < 0].head(5)[['gene', 'log2_fold_change', 'p_adjusted']].to_string(index=False)}
        
        2. BIOLOGICAL PATHWAYS (Manual Literature Review Required)
        -----------------------------------------------------------
        Key pathways to investigate:
        - Inflammatory response
        - Extracellular matrix remodeling
        - Oxidative stress response
        - Immune cell signaling
        
        3. LITERATURE VALIDATION STEPS
        -------------------------------
        a) Search PubMed for each top gene + "COPD"
        b) Check GeneCards (https://www.genecards.org/)
        c) Verify protein function in UniProt
        d) Look for known disease associations
        
        4. CLINICAL RELEVANCE
        ---------------------
        - Potential biomarkers for early detection
        - Therapeutic target identification
        - Patient stratification markers
        """
        
        print(summary)
        
        # Save to file
        with open('./results/biological_interpretation.txt', 'w') as f:
            f.write(summary)
        
        return summary

# Usage
interpreter = BiologicalInterpreter(feature_importance, de_results)
top_genes_report = interpreter.create_top_genes_report(top_n=20)
interpretation_summary = interpreter.create_interpretation_summary()
